# 추가 백엔드-3: FCT FAIL 1/2/3+ 분석 (Jupyter Notebook)
- 사양서(2026-01-29 추가 백엔드-3 설계) 기준으로 작성
- 입력: `prod_day(YYYYMMDD)`, `shift_type(day|night)`
- 출력: 1회 FAIL / 2회 FAIL / 3회 이상 FAIL 별 DataFrame 3개


In [1]:
# (필요 시) 설치
# !pip -q install sqlalchemy psycopg2-binary pandas python-dateutil

import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta, time as dtime
from zoneinfo import ZoneInfo


In [2]:
# =========================
# 1) DB 설정 (사양 그대로)
# =========================
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "leejangwoo1!",
}

def make_engine():
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    # 노트북: 풀 최소화(상시 연결 1개 수준)
    return create_engine(url, pool_size=1, max_overflow=0, pool_pre_ping=True)


In [3]:
# =========================
# 2) 주/야간 시간창 (KST 로컬 기준)
# =========================
KST = ZoneInfo("Asia/Seoul")

def shift_window_kst(prod_day: str, shift_type: str) -> tuple[datetime, datetime]:
    """
    prod_day: 'YYYYMMDD'
    shift_type: 'day' | 'night'
    - day   : [D] 08:30:00 ~ 20:29:59 (inclusive)
    - night : [D] 20:30:00 ~ [D+1] 08:29:59 (inclusive)
    """
    y = int(prod_day[0:4]); m = int(prod_day[4:6]); d = int(prod_day[6:8])
    base = datetime(y, m, d, tzinfo=KST)
    if shift_type.lower() == "day":
        start = base.replace(hour=8, minute=30, second=0, microsecond=0)
        end   = base.replace(hour=20, minute=29, second=59, microsecond=0)
    elif shift_type.lower() == "night":
        start = base.replace(hour=20, minute=30, second=0, microsecond=0)
        end   = (base + timedelta(days=1)).replace(hour=8, minute=29, second=59, microsecond=0)
    else:
        raise ValueError("shift_type must be 'day' or 'night'")
    return start, end


In [4]:
# =========================
# 3) PN/Remark 매핑 (barcode 18번째 문자 key)
#    - g_production_film.remark_info 테이블을 사용
# =========================
# 사양서 예시:
# key(C,P,1,N,J,S) -> pn / remark(PD|Non-PD)
#
# 실제 분석에서는 DB의 remark_info를 조회하여 매핑을 구성한다.
def load_remark_map(engine) -> dict:
    sql = text("""
        SELECT barcode_information AS key_char, pn, remark
        FROM g_production_film.remark_info
    """)
    df = pd.read_sql(sql, engine)
    m = {}
    for _, r in df.iterrows():
        k = str(r["key_char"]).strip()
        if k and k.lower() != "nan":
            m[k] = {"pn": str(r["pn"]).strip(), "remark": str(r["remark"]).strip()}
    return m


In [5]:
# =========================
# 4) 핵심 쿼리: FAIL 로그 조회 (시간창 필터 + 18번째 문자로 PN join)
# =========================
def fetch_fail_rows(engine, start_dt: datetime, end_dt: datetime) -> pd.DataFrame:
    """a2_fct_table.fct_table에서 지정 시간창 내 FAIL 행만."""
    # end_day(text: YYYYMMDD) + end_time(text: HHMMSS)를 timestamp로 캐스팅
    # end_time이 혹시 길이가 6이 아니면 lpad로 보정(안전장치)
    sql = text("""
        WITH base AS (
            SELECT
                ft.station,
                ft.barcode_information,
                ft.step_description,
                ft.result,
                ft.end_day,
                ft.end_time,
                to_timestamp(ft.end_day || lpad(ft.end_time, 6, '0'), 'YYYYMMDDHH24MISS') AS end_ts,
                substring(ft.barcode_information from 18 for 1) AS key_char
            FROM a2_fct_table.fct_table ft
            WHERE ft.result = 'FAIL'
        )
        SELECT
            b.station,
            b.barcode_information,
            b.step_description,
            b.end_day,
            b.end_time,
            b.end_ts,
            b.key_char,
            ri.pn,
            ri.remark
        FROM base b
        LEFT JOIN g_production_film.remark_info ri
            ON b.key_char = ri.barcode_information
        WHERE b.end_ts >= :start_ts
          AND b.end_ts <= :end_ts
    """)
    params = {
        # to_timestamp 결과는 보통 timezone 없는 timestamp이므로 naive datetime으로 전달
        "start_ts": start_dt.replace(tzinfo=None),
        "end_ts": end_dt.replace(tzinfo=None),
    }
    df = pd.read_sql(sql, engine, params=params)
    return df


In [6]:
# =========================
# 5) FAIL 1회/2회/3회+ 분류 + step_description 별 집계
# =========================
def classify_and_aggregate(df_fail: pd.DataFrame, prod_day: str, shift_type: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    - 동일 prod_day/shift window 내에서 barcode_information 별 FAIL 발생 횟수를 계산
    - FAIL 횟수 bucket(1,2,>=3)별로 step_description 집계
    - 출력 스키마는 사양서 형식에 맞춤
    """
    cols1 = ["prod_day","shift_type","pn","station","1st_FAIL_step_description","count","updated_at"]
    cols2 = ["prod_day","shift_type","pn","station","2st_FAIL_step_description","count","updated_at"]
    cols3 = ["prod_day","shift_type","pn","station","3st_over_FAIL_step_description","count","updated_at"]

    if df_fail.empty:
        return (pd.DataFrame(columns=cols1), pd.DataFrame(columns=cols2), pd.DataFrame(columns=cols3))

    # barcode별 FAIL 횟수
    fail_cnt = df_fail.groupby("barcode_information").size().rename("fail_n").reset_index()
    dfx = df_fail.merge(fail_cnt, on="barcode_information", how="left")

    # updated_at: dataframe 생성 시각(KST)
    updated_at = datetime.now(tz=KST)

    def _agg(bucket_df: pd.DataFrame, step_col: str) -> pd.DataFrame:
        if bucket_df.empty:
            if step_col == "1st_FAIL_step_description":
                return pd.DataFrame(columns=cols1)
            if step_col == "2st_FAIL_step_description":
                return pd.DataFrame(columns=cols2)
            return pd.DataFrame(columns=cols3)

        g = (bucket_df
             .groupby(["pn","station","step_description"], dropna=False)
             .size()
             .rename("count")
             .reset_index())

        g.insert(0, "shift_type", shift_type)
        g.insert(0, "prod_day", prod_day)
        g["updated_at"] = updated_at

        g = g.rename(columns={"step_description": step_col})
        return g[["prod_day","shift_type","pn","station",step_col,"count","updated_at"]]

    df1 = _agg(dfx[dfx["fail_n"] == 1].copy(), "1st_FAIL_step_description")
    df2 = _agg(dfx[dfx["fail_n"] == 2].copy(), "2st_FAIL_step_description")
    df3 = _agg(dfx[dfx["fail_n"] >= 3].copy(), "3st_over_FAIL_step_description")

    return df1, df2, df3


In [7]:
# =========================
# 6) 실행 함수
# =========================
def run_backend3(prod_day: str, shift_type: str):
    engine = make_engine()
    start_dt, end_dt = shift_window_kst(prod_day, shift_type)

    print(f"[WINDOW] {prod_day} {shift_type}  {start_dt} ~ {end_dt} (KST)")

    df_fail = fetch_fail_rows(engine, start_dt, end_dt)

    print(f"[RAW FAIL ROWS] n={len(df_fail)}")
    if not df_fail.empty:
        display(df_fail.head(10))

    df1, df2, df3 = classify_and_aggregate(df_fail, prod_day, shift_type)

    return df1, df2, df3


In [9]:
# =========================
# 7) 예시 실행
# =========================
prod_day = "20260127"
shift_type = "day"   # or "night"
df_1st, df_2st, df_3p = run_backend3(prod_day, shift_type)
display(df_1st)
display(df_2st)
display(df_3p)


[WINDOW] 20260127 day  2026-01-27 08:30:00+09:00 ~ 2026-01-27 20:29:59+09:00 (KST)
[RAW FAIL ROWS] n=15


,station,barcode_information,step_description,end_day,end_time,end_ts,key_char,pn,remark
0,FCT1,BA1WJ26027503739PC3T-14F014-AC,1.30 Test USB1 benchmark.maxwr(Mbit/s),20260127,112932,2026-01-27 02:29:32+00:00,C,35643009,Non-PD
1,FCT4,BA1WJ26027505504PC3T-14F014-AC,1.22 Test Carplay type-C,20260127,165428,2026-01-27 07:54:28+00:00,C,35643009,Non-PD
2,FCT4,BA1WJ26027505005PC3T-14F014-AC,1.22 Test Carplay type-C,20260127,165604,2026-01-27 07:56:04+00:00,C,35643009,Non-PD
3,FCT4,BA1WJ26027505015PC3T-14F014-AC,1.22 Test Carplay type-C,20260127,165705,2026-01-27 07:57:05+00:00,C,35643009,Non-PD
4,FCT2,BA1WJ26027505761PC3T-14F014-AC,1.08 Test Boston Firmware Version,20260127,173414,2026-01-27 08:34:14+00:00,C,35643009,Non-PD
5,FCT2,BA1WJ26027502968PC3T-14F014-AC,1.08 Test Boston Firmware Version,20260127,094507,2026-01-27 00:45:07+00:00,C,35643009,Non-PD
6,FCT1,BA1WJ23243500751UPC3T-14F014-AC,1.05 Test DIM-GND(ohm),20260127,202117,2026-01-27 11:21:17+00:00,P,35643010,Non-PD
7,FCT2,BA1WJ23243500555UPC3T-14F014-AC,1.05 Test DIM-GND(ohm),20260127,202135,2026-01-27 11:21:35+00:00,P,35643010,Non-PD
8,FCT4,BA1WJ23243500750UPC3T-14F014-AC,1.05 Test DIM-GND(ohm),20260127,202133,2026-01-27 11:21:33+00:00,P,35643010,Non-PD
9,FCT1,BA1WJ25364500011USJ8T-14F014-AF,1.17 Test DIM-GND(ohm),20260127,202254,2026-01-27 11:22:54+00:00,S,35930928,PD


,prod_day,shift_type,pn,station,1st_FAIL_step_description,count,updated_at
0,20260127,day,35643009,FCT1,1.22 Test Carplay type-C,1,2026-01-29 17:45:30.951132+09:00
1,20260127,day,35643009,FCT1,1.25 Test USB2 benchmark.maxrd(Mbit/s),1,2026-01-29 17:45:30.951132+09:00
2,20260127,day,35643009,FCT1,1.30 Test USB1 benchmark.maxwr(Mbit/s),1,2026-01-29 17:45:30.951132+09:00
3,20260127,day,35643009,FCT2,1.08 Test Boston Firmware Version,1,2026-01-29 17:45:30.951132+09:00
4,20260127,day,35643009,FCT4,1.00 Test RGUSB_MiniB(ohm),1,2026-01-29 17:45:30.951132+09:00
5,20260127,day,35643009,FCT4,1.22 Test Carplay type-C,3,2026-01-29 17:45:30.951132+09:00
6,20260127,day,35643010,FCT1,1.05 Test DIM-GND(ohm),1,2026-01-29 17:45:30.951132+09:00
7,20260127,day,35643010,FCT2,1.05 Test DIM-GND(ohm),1,2026-01-29 17:45:30.951132+09:00
8,20260127,day,35643010,FCT4,1.05 Test DIM-GND(ohm),1,2026-01-29 17:45:30.951132+09:00
9,20260127,day,35930928,FCT1,1.17 Test DIM-GND(ohm),1,2026-01-29 17:45:30.951132+09:00


,prod_day,shift_type,pn,station,2st_FAIL_step_description,count,updated_at
0,20260127,day,35643009,FCT2,1.08 Test Boston Firmware Version,1,2026-01-29 17:45:30.951132+09:00
1,20260127,day,35643009,FCT3,1.08 Test Boston Firmware Version,1,2026-01-29 17:45:30.951132+09:00


,prod_day,shift_type,pn,station,3st_over_FAIL_step_description,count,updated_at


## 출력 정의
- `df_1st` : prod_day/shift_type/pn/station/1st_FAIL_step_description 별 count
- `df_2st` : prod_day/shift_type/pn/station/2st_FAIL_step_description 별 count
- `df_3p`  : prod_day/shift_type/pn/station/3st_over_FAIL_step_description 별 count

※ `updated_at`은 DataFrame 생성/갱신 시각(KST)입니다.
